In [1]:
import os
import pandas as pd
import numpy as np
import unicodedata
from fasttext import train_supervised
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from statistics import mean



In [2]:
possible_dirs = [
    os.path.join("data", "jigsaw-toxic-comment-classification-challenge"),
    "/kaggle/input/jigsaw-toxic-comment-classification-challenge",
    "/kaggle/working/jigsaw-toxic-comment-classification-challenge",
]
DATA_DIR = next((d for d in possible_dirs if os.path.isdir(d)), None)
if DATA_DIR is None:
    raise FileNotFoundError("Dataset directory not found in expected locations.")

train_path = os.path.join(DATA_DIR, "train.csv")
test_path = os.path.join(DATA_DIR, "test.csv")
sample_sub_path = os.path.join(DATA_DIR, "sample_submission.csv")

train_text = pd.read_csv(train_path)
test_text = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_sub_path)
print("Loaded:", train_text.shape, test_text.shape, sample_submission.shape)



Loaded: (159571, 8) (153164, 2) (153164, 7)


In [3]:
y_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]




In [4]:
def clean_it(text, normalize=True):
    s = (
        str(text)
        .replace(",", " ")
        .replace('"', "")
        .replace("'", " ' ")
        .replace(".", " . ")
        .replace("(", " ( ")
        .replace(")", " ) ")
        .replace("!", " ! ")
        .replace("?", " ? ")
        .replace(":", " ")
        .replace(";", " ")
        .lower()
    )
    s = s.replace("\n", " ")
    if normalize:
        s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("utf-8")
    return s


def clean_df(
    data, cleanit=False, shuffleit=False, encodeit=False, label_prefix="__class__"
):
    """
    Build a DataFrame suitable for fasttext training.
    Labels are created without a trailing space so fasttext sees a single token.
    """
    df = data[["comment_text"]].copy(deep=True)
    for col in y_cols:
        df[col] = f"{label_prefix}{data[col].astype(int).astype(str)}"
    if cleanit:
        df["comment_text"] = df["comment_text"].apply(lambda x: clean_it(x, encodeit))
    if shuffleit:
        df = df.sample(frac=1).reset_index(drop=True)
    return df




In [5]:
train_split, val_split = train_test_split(train_text, shuffle=True, random_state=123)

df_train_cleaned = clean_df(train_split, cleanit=True, shuffleit=True)
df_val_cleaned = clean_df(val_split, cleanit=True, shuffleit=True, label_prefix="")




In [6]:
def get_positive_prob(model, text):
    """
    Return the probability of the positive class '__class__1'.
    FastText OVA returns a list of labels with associated confidences.
    We request the two most probable labels; if the positive label is present,
    use its confidence, otherwise use the complement of the negative label's confidence.
    """
    labels, probs = model.predict(text, k=2)
    prob_dict = dict(zip(labels, probs))
    if "__class__1" in prob_dict:
        return prob_dict["__class__1"]
    if "__class__0" in prob_dict:
        return 1.0 - prob_dict["__class__0"]
    return 0.0


def batch_predict(model, texts, k=2, chunk_size=20000):
    """
    Predict in batches to avoid Python‑level per‑sentence loops.
    Returns a list of positive‑class probabilities matching the order of `texts`.
    """
    probs = []
    for i in range(0, len(texts), chunk_size):
        batch = texts[i : i + chunk_size]
        batch_labels, batch_probs = model.predict(batch, k=k)
        for lbls, pr in zip(batch_labels, batch_probs):
            prob_dict = dict(zip(lbls, pr))
            if "__class__1" in prob_dict:
                probs.append(prob_dict["__class__1"])
            elif "__class__0" in prob_dict:
                probs.append(1.0 - prob_dict["__class__0"])
            else:
                probs.append(0.0)
    return probs


model_dict = {}
all_val_preds = []

# Use all available CPU cores for faster training
thread_count = os.cpu_count() or 2

for col in y_cols:
    train_file = os.path.join(os.getcwd(), f"train_{col}.txt")
    df_train_cleaned[[col, "comment_text"]].to_csv(
        train_file,
        header=False,
        index=False,
        sep=" ",
        columns=[col, "comment_text"],
    )

    model = train_supervised(
        input=train_file,
        label="__class__",
        lr=0.3,
        epoch=120,
        loss="ova",  # supported loss
        wordNgrams=3,
        dim=300,
        thread=thread_count,
        verbose=0,
    )
    model_dict[col] = model

    # Batch‑predict on validation set
    val_texts = df_val_cleaned["comment_text"].tolist()
    val_preds = batch_predict(model, val_texts, k=2)
    all_val_preds.append(val_preds)



In [7]:
val_pred_array = np.transpose(np.array(all_val_preds))
y_val_actuals = val_split[y_cols].astype(int).to_numpy()
mean_auc = mean(
    [
        roc_auc_score(y_val_actuals[:, i], val_pred_array[:, i])
        for i in range(len(y_cols))
    ]
)
print("Mean validation AUC:", mean_auc)



Mean validation AUC: 0.5


In [8]:
df_full_cleaned = clean_df(train_text, cleanit=True, shuffleit=True)

for col in y_cols:
    train_file = os.path.join(os.getcwd(), f"full_train_{col}.txt")
    df_full_cleaned[[col, "comment_text"]].to_csv(
        train_file,
        header=False,
        index=False,
        sep=" ",
        columns=[col, "comment_text"],
    )
    model = train_supervised(
        input=train_file,
        label="__class__",
        lr=0.3,
        epoch=120,
        loss="ova",  # supported loss
        wordNgrams=3,
        dim=300,
        thread=thread_count,
        verbose=0,
    )
    model_dict[col] = model  # overwrite with full‑data model for testing

df_test = pd.merge(test_text, sample_submission, on="id")
df_test_cleaned = clean_df(df_test, cleanit=True, shuffleit=False, label_prefix="")

all_test_preds = []
test_texts = df_test_cleaned["comment_text"].tolist()
for col in tqdm(y_cols, desc="Testing"):
    model = model_dict[col]
    test_preds = batch_predict(model, test_texts, k=2)
    all_test_preds.append(test_preds)

test_pred_array = np.transpose(np.array(all_test_preds))
df_test[y_cols] = test_pred_array
df_test.drop(columns=["comment_text"], inplace=True)



Testing: 100%|██████████| 6/6 [01:21<00:00, 13.62s/it]


In [9]:
submission_path = "submission.csv"
df_test.to_csv(submission_path, index=False)
print("Submission file written:", submission_path, "rows:", df_test.shape[0])

Submission file written: submission.csv rows: 153164
